In [33]:
"""
1. Read the file , how can u read it , by opening the file and readlines()
2. perfrom transformation , cleaned into 1 and rejected into 1 
3. logging to be included 
4. JSON output
5. SQl connect """


'\n1. Read the file , how can u read it , by opening the file and readlines()\n2. perfrom transformation , cleaned into 1 and rejected into 1 \n3. logging to be included \n4. JSON output\n5. SQl connect '

In [4]:
# ingestion
""" Opening to file to read """ 

def ingestion(file_path):
    with open(file_path, "r") as file:
        lines= file.readlines()
        return lines[0:] 

#Transformation        
"""Applying the trasnformation logics where you can transform the data into cleaned and rejected data sets and apply logics  """

import logging
import datetime
logging.basicConfig(
    level=logging.INFO,
    format = '%(asctime)s - %(levelname)s - %(message)s')

def transformation(file):
    logging.info("The tranformation process has started")
    clean = []
    rejected = []
    for line in file:
        try:
            user_id , name, age , country = line.strip().split(",")
            age = int(age)
            if not name or age > 100 or age < 0:
                raise ValueError("Invalid record")

            clean.append({
                "id" : user_id,
                "name" : name,
                "age": age,
                "country" : country })
            logging.info(f"valid : {user_id}")

        except Exception as e:
            rejected.append(line.strip())
            logging.error(f"Rejected :{line.strip()} | Reason {e}")
            
    return  clean, rejected

def dq_report(clean, rejected):
    report = { "Total_records" : len(clean) + len(rejected) ,
               "clean_record_count" : len(clean),
               "rejected_record_count" : len(rejected),
               "rejected_rate%" : round(len(rejected) / (len(clean)+len(rejected))*100 , 2),
               #"generated_at" : str(datetime.now())
             }
    with open("DQ_report.json", "w") as f:
        json.dump(report,f , indent=4)
    logging.info (f"DQ report was saved - {report['rejected_rate%']}% rejection rate")
    return report               

    
               
""" converting to json file , basically taking list as input and converting them using json.dump()"""

import json
def load_json(clean_data):
    with open("cleaned.json" , "w") as file:
        json.dump(clean_data , file , indent=4)

# adding Sql connections:
import sqlite3
def load_sql(clean_data):
    conn=sqlite3.connect("customer.db")
    cur=conn.cursor()
    cur.execute("""
        create table if not exists customers(
          id text, 
          name  text,
          age integer, 
          country text )
    """)
    for row in clean_data:
        cur.execute(
            "insert into customers values(?,?,?,?)",
            (row["id"],row["name"], row["age"],row["country"] )
        )
    conn.commit()
    conn.close()
    
def table_validation():
    conn = sqlite3.connect("customer.db")
    cur= conn.cursor()
    cur.execute("select count(*) from customers")
    print("total Records" , cur.fetchone()[0]) 
    cur.execute("select * from customers")
    rows =cur.fetchall()
    for row in rows:
        print(row)
    conn.close()
    print("Records have been added")
    
    
    
def main():
    data = ingestion("/data/customers.txt")
    clean , rejected = transformation(data)
    dq_report(clean, rejected)
    load_json(clean)
    load_sql(clean)
    table_validation()
    print("pipeline completed")
    print("valid length" , len(clean))
    print("invlaid length" , len(rejected))

if __name__ == "__main__":
    main()
    

FileNotFoundError: [Errno 2] No such file or directory: '/data/customers.txt'

In [20]:
"""Understanding the logic : 

# transformation : 
Here , for every line in the file: 
line.strip() --> will remove extra space , newline characters front/back 
"111 , sur , 22 , ind\n"-->  "111 , sur , 22 , ind" 

line.split(,) --> wlill split the file based on ,(comma)
split(',') ---> "111 , sur , 22 , ind"  --> ["111", "sur" , "22" , "ind" ] , everything is still str .
    
user_id , name, age , country =  ["111", "sur" , "22" , "ind" ] --> iterable unpacking : works for tuple , list and string 
then unpacking into variables :
    user_id = "101"
    name = "sur"
    age = "22"
    country = "ind"
if age:
    age = int(age)
else: 
    age = None
    ---> Always convert to the numeric values  at the beginnig itself : like , age , user_id 
        
======================================================================================================
Explanation: 
        
   for line in file:
        try:
            user_id , name, age , country = line.strip().split(",")
            age = int(age)
            user_id = int(user_id)
            if not name or age > 100 or age < 0:
                raise ValueError("Invalid record")

            clean.append({
                "id" : user_id,
                "name" : name,
                "age": age,
                "country" : country })
            logging.info(f"valid : {user_id}")

        except Exception as e:
            rejected.append(line.strip())
            logging.error(f"Rejected :{line.strip()} | Reason {e}")

    """         

SyntaxError: invalid syntax (3991014011.py, line 1)

In [ ]:
"""# for the Db connection 

conn=sqlite3.connect("customers.db") -->creates if not exists
cur =conn.cursor()----> gets the sql executor to execute the queries
cur.execute(
    """create table if not exists customers (
         id  int , 
         name text, 
         country text, 
         age int
    """) --------------------> creates the table schema 
for row in clean_data: ---> for every row in the clen dsata list 
    cur.execute("insert into customer values(?,?,?,?) ",   
                (row["id"], row["name"], row["country"] , row["age"]))" -----------> ? this is the placeholder , we can use {id} --> but
                   # this results in format issues,confusions so ? this is good practice. 
                   # row["id"] --> should use "" bocz the clean_data is a dictionary
                   # (row["id"], row["name"])---> this is a tuple , actual values should be in tuple 
    cur.execute("select count(*) from customers")
    cur.execute("Total Records" , cur.fetchone()[0]) ---> python always return the results in () even for 1 record-->tuple so (5,)->so cur.fetchone()[0]
    
 cur.commit()
 cur.close()
"""    

In [5]:
import os 